# eduACDC tutorial: cluster formation from sulfuric acid and ammonia

This notebook is a hands-on introduction to **eduACDC**, an educational Python version of the Atmospheric Cluster Dynamics Code (ACDC). ACDC simulates how vapour molecules collide, stick together and evaporate, and how often the resulting molecular clusters grow large enough to become new particles.

You do not need to know Python to use this notebook. Read the text, run the code cells in order, and look at the results. Later you can change a few clearly marked numbers and run again. Working through the notebook takes about 30–45 minutes.

**What you will do**

1. Load a ready-made description of the sulfuric acid–ammonia–water system.
2. Look at the cluster thermodynamics: formation free energies and evaporation rates.
3. Run a time-dependent simulation and watch the clusters form.
4. Compute the steady-state new-particle formation rate $J$ and find the main growth pathway.
5. See how $J$ depends on acid concentration and temperature.

## 0. How to use a Jupyter notebook

A notebook consists of **cells**. This cell is a *text cell*. The grey cells with `[ ]:` next to them are *code cells*, which contain Python code.

- To run a code cell, click it and press **Shift + Enter** (or click the ▶ button). The output appears below the cell, and the next cell is selected, so you can keep pressing Shift + Enter to go through the notebook.
- Run the cells **from top to bottom**. Later cells use results from earlier ones. If you get an error such as `NameError: name '...' is not defined`, you have most likely skipped a cell above; go back and run it.
- While a cell runs, it shows `[*]`; when it has finished, a number appears, e.g. `[5]`.
- If things get confused, restart from scratch: in Google Colab choose *Runtime → Restart session and run all*; in JupyterLab choose *Kernel → Restart Kernel and Run All Cells*.
- Lines starting with `#` in code cells are comments for you to read; Python ignores them.
- Places where you are invited to change values are marked with `# <-- try changing this`.

Nothing you do here can break anything permanently. If you have made a mess, you can always restart, or download a fresh copy of the notebook.

## 1. Setup

The next cell prepares everything. In Google Colab it first installs eduACDC (this takes about a minute the first time). If you run the notebook on your own computer, eduACDC must already be installed; see `TUTORIAL_SETUP.md` in the same folder. The cell then loads the tools we need and fetches the configuration file of the chemical system.

Run it now with **Shift + Enter**. Messages about installing packages are normal.

In [ ]:
import sys

# The eduACDC version this tutorial was tested with (a fixed commit on GitHub).
EDUACDC_COMMIT = "7b4aca04a2b96d98493dbb5e9b3cd6031e414666"

# Install eduACDC if we are running in Google Colab.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install --quiet "eduacdc @ git+https://github.com/edahelsinki/eduACDC.git@{EDUACDC_COMMIT}"

import logging
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from eduacdc.io import InputParser
from eduacdc.simulation import Simulation
from eduacdc.analysis import (
    plot_reference_deltag_surface,
    plot_act_deltag_surface,
    plot_overall_evaporation_rate_surface,
    plot_formation_rates,
    plot_final_concentrations,
    plot_flux_tracking,
    plot_final_outflux,
)

# Show only warnings and errors from eduACDC, not every detail of its work.
logging.getLogger("eduacdc").setLevel(logging.WARNING)


def get_config_file(name):
    # Use the configuration file in this folder if it exists, otherwise download it from GitHub.
    path = Path(name)
    if not path.exists():
        url = f"https://raw.githubusercontent.com/edahelsinki/eduACDC/{EDUACDC_COMMIT}/examples/{name}"
        urllib.request.urlretrieve(url, path)
    return path


print("Setup done.")

## 2. The chemical system

eduACDC reads the description of a chemical system from a configuration file in YAML format, which is a plain text file. We use `ANW.yaml`, which describes clusters of sulfuric acid (`A`) and ammonia (`N`) in the presence of water (`W`). Let us look at the beginning of the file. (The first comment line in the file mentions dimethylamine; that is a leftover typo, the file is about ammonia.)

In [ ]:
config_file = get_config_file("ANW.yaml")

lines = config_file.read_text().splitlines()
print("\n".join(lines[:36]))
print("   ... (more energies follow) ...")

The file has the following parts:

- **`molecules`**: the molecule types with their names, masses and bulk densities. The masses and densities determine the cluster sizes and hence how often clusters collide.
- **`clusters`**: which clusters are included in the simulation. Here all clusters with 1–4 acid and 1–4 ammonia molecules, plus pure acid and pure ammonia clusters of up to four molecules. A label such as `2A1N` means a cluster of two sulfuric acid molecules and one ammonia molecule.
- **`conditions`**: the relative humidity (here 50 %).
- **`cluster_properties`**: the thermodynamic data, i.e. the formation enthalpy $\Delta H$ (kcal/mol) and entropy $\Delta S$ (cal/(mol K)) of each cluster, computed with quantum chemistry. From these the code obtains the formation free energy $\Delta G = \Delta H - T\Delta S$ at any temperature $T$. Water is not tracked as a separate molecule; clusters containing water (`1A1W` etc.) are used to compute an average over the hydrates at the given humidity.
- **`equations_configuration`** (at the end of the file): the *outgrowth rules*. A cluster that grows past the simulated set to `4A5N`, `5A4N` or `5A5N` is considered stable enough to keep growing; it leaves the simulation and is counted as a newly formed particle.

> **TODO (physics check):** Check the description of hydrate averaging and of the outgrowth rule (in particular, what happens to clusters that grow out of the simulated set but do not satisfy the rule).

Next we turn the file into Python objects: the `system` (molecules, clusters and their properties) and the `equations` (the differential equations describing all collisions and evaporations). Then we combine them into a `Simulation`.

In [ ]:
parser = InputParser()
system = parser.get_system_from_yaml(config_file)
equations = parser.get_equations_from_yaml(config_file, system.clusters)
sim = Simulation(system, equations)

print("Clusters in the simulation:")
print(system.clusters.get_labels())

### What the simulation computes

For each cluster $i$, ACDC solves an equation for how its concentration $C_i$ changes in time (the *birth–death equations*):

$$
\frac{dC_i}{dt} = \tfrac12 \sum_{j+k \to i} \beta_{j,k} C_j C_k + \sum_j \gamma_{i+j \to i,j} C_{i+j} - \sum_j \beta_{i,j} C_i C_j - \tfrac12 \sum_{j+k=i} \gamma_{i \to j,k} C_i + Q_i - S_i ,
$$

where $\beta$ are collision coefficients (how often two clusters meet, from kinetic gas theory), $\gamma$ are evaporation coefficients (how often a cluster falls apart), $Q_i$ is a source and $S_i$ represents external losses. The evaporation coefficients follow from the collision coefficients and the formation free energies $\Delta G$: the more negative $\Delta G$, the more stable the cluster and the slower it evaporates.

The new-particle formation rate $J$ is the rate at which clusters leave the simulated set through the outgrowth rules, in units of cm⁻³ s⁻¹ (new particles per cubic centimetre per second).

For details, see McGrath et al. (2012), *Atmospheric Cluster Dynamics Code: a flexible method for solution of the birth–death equations*, https://doi.org/10.5194/acp-12-2345-2012.

> **TODO (physics check):** Check the equation (factors ½, notation) and the short description of β, γ and J.

## 3. Choosing the conditions

Here we set the temperature and the vapour concentrations. The acid and ammonia monomers are kept at constant concentrations, as if the vapours were continuously replenished.

Concentrations are given as text with units, e.g. `"1e7 cm^-3"` (molecules per cubic centimetre; `1e7` means $10^7$) or `"100 ppt"` (parts per trillion of the gas). These values give clearly visible particle formation; later you can try other values.

In [ ]:
TEMPERATURE = 278.15         # K  <-- try changing this (e.g. 258.15 or 298.15)
ACID = "1e7 cm^-3"           # sulfuric acid monomer  <-- try changing this
AMMONIA = "100 ppt"          # ammonia monomer  <-- try changing this

initial_conditions = {
    "1A": {"type": "constant", "value": ACID},
    "1N": {"type": "constant", "value": AMMONIA},
}

## 4. Thermodynamics: how stable are the clusters?

Before simulating anything, let us look at the input data. The plots below are maps over cluster composition: the horizontal axis is the number of acid molecules, the vertical axis the number of ammonia molecules, so each square is one cluster.

### 4.1 Formation free energy at reference conditions

The first map shows the formation free energy $\Delta G_\text{ref}$ of each cluster from its monomers at the reference pressure of 1 atm, in kcal/mol. Strongly negative values (blue) mean strongly bound clusters. Notice that clusters with roughly equal numbers of acid and ammonia molecules are the most stable: ammonia is a base and binds to the acid.

In [ ]:
# Make sure the system is at our chosen temperature (some later cells change it).
system.update_conditions(temperature=TEMPERATURE)

ax = plot_reference_deltag_surface(system, "A", "N", max_A=4, max_B=4)

### 4.2 Free energy at the actual vapour concentrations

At atmospheric concentrations the vapours are far below 1 atm, which makes cluster formation much less favourable. The second map shows the *actual* free energy
$$\Delta G = \Delta G_\text{ref} - k_B T \sum_i n_i \ln\frac{p_i}{p_\text{ref}},$$
where $n_i$ is the number of molecules of type $i$ in the cluster and $p_i$ the partial pressure of that vapour. (With this convention the monomers do not have zero free energy.) Clusters have to grow over a free energy barrier, and clusters beyond the barrier tend to keep growing.

> **TODO (physics check):** Check the convention of the actual ΔG (monomers are not at zero; is this the intended presentation?) and the barrier interpretation. See also Olenius et al. (2013), https://doi.org/10.1063/1.4819024.

In [ ]:
monomer_concentrations = {"A": ACID, "N": AMMONIA}
ax = plot_act_deltag_surface(system, "A", "N", max_A=4, max_B=4,
                             monomer_concentrations=monomer_concentrations)

### 4.3 Evaporation rates

This map shows the total evaporation rate of each cluster, in s⁻¹ (how many times per second the cluster loses a molecule). Compare it with the collision rate of a cluster with acid monomers, which at $10^7$ cm⁻³ is about 0.005 s⁻¹ in this model (one collision every few minutes): clusters that evaporate much faster than they collide with acid do not survive to grow.

> **TODO (physics check):** Check this comparison (collision frequency computed from the model's collision coefficients, 3.7–5.1·10⁻¹⁰ cm³ s⁻¹ for acid monomer collisions at 278 K).

In [ ]:
ax = plot_overall_evaporation_rate_surface(system, "A", "N", max_A=4, max_B=4)

## 5. Time-dependent simulation

Now we start from a situation where only acid and ammonia monomers exist, and follow how the clusters build up over 10 000 seconds (about three hours). The result is stored in `time_results`.

In [ ]:
time_results = sim.run(initial_conditions, t_span=(0, 1e4), temperature=TEMPERATURE)
print("Number of time steps:", len(time_results.time))

Let us plot the concentrations of a few clusters along the main growth route. Both axes are logarithmic. Small clusters reach their final concentration within seconds, larger ones take longer. Concentrations are stored in m⁻³, so we multiply by $10^{-6}$ to get cm⁻³.

In [ ]:
clusters_to_plot = ["2A", "2A1N", "2A2N", "3A2N", "3A3N", "4A4N"]   # <-- try adding e.g. "1A1N"

fig, ax = plt.subplots(figsize=(8, 5))
largest = 0
for label in clusters_to_plot:
    i = system.clusters.get_index_by_label(label)
    concentration_cm3 = time_results.concentrations[:, i] * 1e-6
    ax.plot(time_results.time, concentration_cm3, label=label)
    largest = max(largest, concentration_cm3.max())
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e-2, 1e4)                        # show times from 0.01 s to 10 000 s
ax.set_ylim(largest * 1e-12, largest * 10)    # show 13 orders of magnitude
ax.set_xlabel("Time (s)")
ax.set_ylabel("Concentration (cm$^{-3}$)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

The formation rate $J$ (the flux of clusters out of the simulated set) builds up in the same way and then levels off at a constant value. The very first instants (below about $10^{-2}$ s) are left out because $J$ is still essentially zero there.

In [ ]:
ax = plot_formation_rates(time_results)
J_max = time_results.get_formation_rates("cm^-3 s^-1").max()
ax.set_xlim(1e-2, 1e4)
ax.set_ylim(J_max * 1e-10, J_max * 10)
plt.show()

## 6. Steady state and the formation rate

Because the monomer concentrations are held constant, the system approaches a **steady state** in which all cluster concentrations, and $J$, no longer change. Often this is all we need, and eduACDC can compute it directly. We also ask it to compute the fluxes between clusters, which we use in Section 7.

In [ ]:
steady = sim.run_steady_state(
    initial_conditions,
    temperature=TEMPERATURE,
    compute_net_fluxes=True,      # needed for the pathway analysis in Section 7
    compute_final_outflux=True,   # needed for the pie chart in Section 7
    calc_final_sources=False,     # workaround for a bug in this eduACDC version
)

J = steady.get_final_formation_rate("cm^-3 s^-1")
print(f"Steady-state formation rate J = {J:.3g} cm^-3 s^-1")

The steady-state value should agree with the end of the time-dependent run in Section 5. The bar chart shows the steady-state concentrations of all clusters (logarithmic scale). Note how many orders of magnitude they span.

In [ ]:
ax = plot_final_concentrations(steady, output_units="cm^-3", figsize=(8, 7))

## 7. Growth pathways: how do the clusters grow?

Which route do the clusters take on their way out of the simulated set? The next plot shows the cluster compositions as grid points (acid horizontally, ammonia vertically) and arrows for the main net fluxes between them. The colour and thickness of an arrow indicate the size of the flux, and the dashed line marks the edge of the simulated set. The dominant route is shown in red.

> **TODO (physics check):** Check the description of the flux-tracking plot (what exactly is drawn, and the meaning of the threshold).

In [ ]:
ax = plot_flux_tracking(steady, threshold=0.01)

The growth mainly proceeds through clusters with roughly equal numbers of acid and ammonia molecules, as the free energies in Section 4 suggested. The pie chart shows which collisions take clusters out of the simulated set, i.e. which collisions produce the new particles.

In [ ]:
ax = plot_final_outflux(steady)

## 8. How does J depend on acid concentration and temperature?

A typical use of ACDC is to compute $J$ for many conditions. The cell below loops over five acid concentrations and three temperatures, computes the steady state for each combination, and plots the result. It takes a few seconds.

In [ ]:
acid_values = [1e6, 3e6, 1e7, 3e7, 1e8]          # cm^-3
temperatures = [258.15, 278.15, 298.15]          # K

fig, ax = plt.subplots(figsize=(8, 5))
for T in temperatures:
    J_values = []
    for acid in acid_values:
        conditions = {
            "1A": {"type": "constant", "value": f"{acid} cm^-3"},
            "1N": {"type": "constant", "value": AMMONIA},
        }
        result = sim.run_steady_state(conditions, temperature=T)
        J_values.append(result.get_final_formation_rate("cm^-3 s^-1"))
    ax.plot(acid_values, J_values, "o-", label=f"T = {T} K")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Sulfuric acid concentration (cm$^{-3}$)")
ax.set_ylabel("Formation rate J (cm$^{-3}$ s$^{-1}$)")
ax.set_title(f"Ammonia {AMMONIA}")
ax.legend()
ax.grid(alpha=0.3)

Two things stand out. First, $J$ increases steeply with the acid concentration. On a log–log plot the slope tells roughly how many acid molecules the critical cluster contains: a slope of 4 means that $J$ grows as the fourth power of the acid concentration. Second, $J$ drops by many orders of magnitude when the temperature rises, because clusters evaporate much faster at higher temperatures.

Note that the loop leaves the system at the last temperature in the list. That is why every cell in this notebook passes the temperature explicitly.

> **TODO (physics check):** Check the interpretation of the slope (first nucleation theorem; caveats when the formation is not barrier-limited) and the temperature explanation.

## 9. Exercises

1. **Change the conditions.** Go back to Section 3, change `AMMONIA` to `"10 ppt"` or `"1000 ppt"`, and rerun the notebook from Section 3 onwards (or use *Run all*). How does $J$ change? Does the growth pathway change?
2. **Change the temperature.** Set `TEMPERATURE = 258.15` (−15 °C) and rerun. What happens to the free energy maps and to $J$?
3. **A stronger base.** Dimethylamine (`D`) is a much stronger base than ammonia. Run the cell below, which repeats the steady-state calculation for the sulfuric acid–dimethylamine system (`ADW.yaml`) with only 1 ppt of dimethylamine. Compare $J$ with the ammonia case.

In [ ]:
config_dma = get_config_file("ADW.yaml")
system_dma = parser.get_system_from_yaml(config_dma)
equations_dma = parser.get_equations_from_yaml(config_dma, system_dma.clusters)
sim_dma = Simulation(system_dma, equations_dma)

conditions_dma = {
    "1A": {"type": "constant", "value": ACID},
    "1D": {"type": "constant", "value": "1 ppt"},   # <-- try changing this
}
steady_dma = sim_dma.run_steady_state(
    conditions_dma, temperature=TEMPERATURE,
    compute_net_fluxes=True, compute_final_outflux=True, calc_final_sources=False,
)
print(f"J (acid-dimethylamine) = {steady_dma.get_final_formation_rate('cm^-3 s^-1'):.3g} cm^-3 s^-1")
ax = plot_flux_tracking(steady_dma, threshold=0.01)

## 10. Where to go from here

- The `examples` folder contains further configuration files, including systems with ions (`AN_neutral_neg_pos.yaml`) and a three-component system. These need an ion source to be specified in the initial conditions.
- External losses (coagulation onto pre-existing particles, chamber wall losses) can be added to the simulations; see `examples/run_simulation.py`.
- The configuration file format is documented in `eduacdc/README.md` in the eduACDC repository: https://github.com/edahelsinki/eduACDC
- The original ACDC code, with references to the method and its applications: https://github.com/tolenius/ACDC

**References**

- McGrath, M. J., Olenius, T., Ortega, I. K., Loukonen, V., Paasonen, P., Kurtén, T., Kulmala, M. and Vehkamäki, H. (2012). Atmospheric Cluster Dynamics Code: a flexible method for solution of the birth–death equations. *Atmospheric Chemistry and Physics* 12, 2345–2355. https://doi.org/10.5194/acp-12-2345-2012
- Olenius, T., Kupiainen-Määttä, O., Ortega, I. K., Kurtén, T. and Vehkamäki, H. (2013). Free energy barrier in the growth of sulfuric acid–ammonia and sulfuric acid–dimethylamine clusters. *The Journal of Chemical Physics* 139, 084312. https://doi.org/10.1063/1.4819024